
# NanoVision — Drug Discovery Analysis (Colab)

This notebook replicates the **Drug Discovery Analysis** workflow from `src/pages/DrugPrediction.tsx` with complete features:
- Load analysis from history (search + select + source mode original/optimized)
- Run sample sync from selected history item
- Full editable inputs (sample name, SMILES, molecular and temporal factors)
- Prediction engine (efficacy/toxicity/multi-factor/decision + ADMET + molecular interaction + toxicity sub-scores + confidence + benchmark)
- 2D chemical diagram with PubChem primary + Cactus fallback
- Run prediction and render full results sections
- Save prediction record history
- Save prediction report as text
- Quick management: delete analysis history item or prediction history item
- Export/import analysis and prediction JSON datasets


In [ ]:

#@title 1) Install dependencies (Colab)
!pip -q install ipywidgets requests pandas pillow numpy


In [ ]:

#@title 2) Imports
import base64
import io
import json
import math
import random
import uuid
from datetime import datetime
from copy import deepcopy

import ipywidgets as widgets
import pandas as pd
import numpy as np
import requests
from PIL import Image as PILImage
from IPython.display import HTML, Markdown, clear_output, display

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


In [ ]:

#@title 3) Helpers + mock analysis history generator (for sample sync)
def clamp(value, low=0, high=100):
    return max(low, min(high, value))

def to_safe_number(value, fallback):
    try:
        n = float(value)
        return n if math.isfinite(n) else fallback
    except Exception:
        return fallback

def risk_band(value):
    return 'LOW' if value < 35 else 'MODERATE' if value < 65 else 'HIGH'

MOCK_COMPOUNDS = [
    {"smiles": "CC(=O)OC1=CC=CC=C1C(=O)O", "molecularWeight": 180.16, "solubility": 3.2},
    {"smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "molecularWeight": 206.28, "solubility": 0.08},
    {"smiles": "CC(=O)NC1=CC=C(O)C=C1", "molecularWeight": 151.16, "solubility": 14.0},
    {"smiles": "CN1CCC[C@H]1C2=CN=CC=C2", "molecularWeight": 162.23, "solubility": 10.5},
    {"smiles": "C1=CC=C(C=C1)C=O", "molecularWeight": 106.12, "solubility": 6.7},
]

def run_mock_analysis(seed=0):
    random.seed(seed)
    c = random.choice(MOCK_COMPOUNDS)

    stability = round(random.uniform(58, 95), 1)
    uniformity = round(random.uniform(62, 98), 1)
    interaction = round(random.uniform(45, 90), 1)
    aggregation = round(random.uniform(0.08, 0.62), 2)
    circularity = round(random.uniform(0.62, 0.95), 3)
    risk = round(clamp(100 - (stability * 0.35 + uniformity * 0.25 + interaction * 0.2 + (100 - aggregation*100) * 0.2)), 1)
    weighted = round(clamp((stability + uniformity + interaction) / 3), 1)
    final = round(clamp(weighted - risk * 0.12 + 16), 1)

    diffusion_t = round(clamp(random.uniform(58, 92)), 1)
    movement_t = round(clamp(random.uniform(56, 90)), 1)
    response_t = round(clamp(random.uniform(52, 88)), 1)

    predicted_eff = round(clamp((interaction * 0.28 + response_t * 0.22 + diffusion_t * 0.2 + movement_t * 0.15 + (100 - risk) * 0.15)), 1)
    pred_tox = round(clamp(risk * 0.7 + (100-circularity*100)*0.3), 1)
    multi = round(clamp(predicted_eff * 0.65 + (100 - pred_tox) * 0.35), 1)

    return {
        'nucleiCount': int(random.uniform(30, 140)),
        'meanArea': round(random.uniform(210, 680), 1),
        'circularity': circularity,
        'aggregationScore': aggregation,
        'stabilityScore': stability,
        'uniformityScore': uniformity,
        'interactionStrength': interaction,
        'screeningDecision': 'Promising Candidate' if final >= 80 else 'Needs Optimization' if final >= 62 else 'Low Performance',
        'screeningMetrics': {
            'riskScore': risk,
            'finalScreeningScore': final,
            'stabilityRisk': round(clamp((100-stability) * 0.6 + aggregation*100*0.4), 1),
            'psnr': round(random.uniform(23, 36), 2),
            'ssim': round(random.uniform(0.76, 0.97), 3),
            'segmentationConfidence': round(random.uniform(78, 98), 1),
            'membraneInteractionScore': interaction,
            'cytotoxicityRisk': round(clamp(aggregation * 90 + (1-circularity)*25), 1),
            'zetaPotentialProxy': round(random.uniform(-34, 10), 1),
            'diffusionCoefficient': round(random.uniform(0.15, 1.05), 3),
            'transportEfficiency': round(clamp((uniformity*0.6 + stability*0.4)), 1),
            'bioavailabilityPrediction': round(clamp((uniformity*0.4 + interaction*0.4 + diffusion_t*0.2)), 1),
            'clusterFormation': round(clamp(aggregation * 100), 1),
            'densityVariation': round(random.uniform(3, 48), 1),
            'particleOverlap': round(clamp(aggregation*80 + (100-circularity*100)*0.25), 1),
            'featureVectorIntegration': round(clamp((interaction + diffusion_t + response_t)/3), 1),
            'weightedScore': weighted,
            'thresholdGap': round(weighted - 62, 2),
            'modelHeadRisk': round(clamp(100 / (1 + math.exp(-(risk - 45) / 8))), 1),
            'smiles': c['smiles'],
            'molecularWeight': round(c['molecularWeight'] + random.uniform(-3, 3), 2),
            'bindingAffinity': round(-5.2 - random.uniform(0, 6.4), 2),
            'solubility': round(clamp(c['solubility'] + random.uniform(-1.2, 1.2), 0.02, 20), 2),
            'cellUptakeRate': round(clamp(random.uniform(45, 90)), 1),
            'proteinInteraction': round(clamp(interaction + random.uniform(0, 10)), 1),
            'targetReceptorBinding': round(clamp(random.uniform(55, 92)), 1),
            'diffusionTrend': diffusion_t,
            'movementTrend': movement_t,
            'responseTrend': response_t,
            'predictedDrugEfficacy': predicted_eff,
            'predictiveToxicityScore': pred_tox,
            'multiCriteriaScreeningScore': multi,
            'pharmacodynamicsIndex': round(clamp((predicted_eff + interaction + response_t)/3), 1),
            'automatedDecision': 'Promising Candidate' if multi >= 72 else 'Needs Optimization' if multi >= 52 else 'Reject',
        }
    }

def make_history_entry(name, seed=0):
    base = run_mock_analysis(seed)
    opt = deepcopy(base)
    opt['screeningMetrics']['riskScore'] = round(clamp(base['screeningMetrics']['riskScore'] * 0.78), 1)
    opt['screeningMetrics']['finalScreeningScore'] = round(clamp(base['screeningMetrics']['finalScreeningScore'] + 9), 1)
    opt['screeningMetrics']['predictedDrugEfficacy'] = round(clamp(base['screeningMetrics']['predictedDrugEfficacy'] + 7), 1)
    opt['screeningMetrics']['predictiveToxicityScore'] = round(clamp(base['screeningMetrics']['predictiveToxicityScore'] * 0.83), 1)

    return {
        'id': str(uuid.uuid4()),
        'createdAt': datetime.utcnow().isoformat() + 'Z',
        'imageName': name,
        'imageData': '',
        'result': base,
        'optimizedResult': opt,
        'optimizedName': name.rsplit('.',1)[0] + '-optimized',
    }


In [ ]:

#@title 4) Prediction engine (port from src/pages/DrugPrediction.tsx)
def compute_prediction(inputs):
    molecularWeight = inputs['molecularWeight']
    bindingAffinity = inputs['bindingAffinity']
    solubility = inputs['solubility']
    cellUptakeRate = inputs['cellUptakeRate']
    proteinInteraction = inputs['proteinInteraction']
    targetReceptorBinding = inputs['targetReceptorBinding']
    diffusionTrend = inputs['diffusionTrend']
    movementTrend = inputs['movementTrend']
    responseTrend = inputs['responseTrend']

    transportEfficiency = clamp((diffusionTrend + movementTrend) / 2)
    predictedBioavailability = clamp(responseTrend * 0.45 + diffusionTrend * 0.35 + cellUptakeRate * 0.2)

    efficacy = clamp(
        targetReceptorBinding * 0.24 +
        proteinInteraction * 0.2 +
        cellUptakeRate * 0.18 +
        diffusionTrend * 0.14 +
        movementTrend * 0.12 +
        responseTrend * 0.12
    )

    overallToxicity = clamp((abs(bindingAffinity) * 5.2 + (100 - solubility * 4) + (100 - cellUptakeRate)) / 3)
    multiFactor = clamp(efficacy * 0.67 + (100 - overallToxicity) * 0.33)
    decision = 'Promising Candidate' if multiFactor >= 72 else 'Needs Optimization' if multiFactor >= 55 else 'Reject'

    lipinskiCompliance = 'PASS' if (molecularWeight <= 500 and abs(bindingAffinity) <= 12 and solubility >= 0.5) else 'FAIL'
    waterSolubilityClass = 'HIGH' if solubility >= 8 else 'MODERATE' if solubility >= 3 else 'LOW'
    cypRisk = 'LOW RISK' if overallToxicity < 35 else 'MODERATE RISK' if overallToxicity < 65 else 'HIGH RISK'
    plasmaProteinBinding = max(15, min(98, 42 + abs(bindingAffinity) * 4.8))
    halfLifeHours = max(1.2, min(24, 2.8 + transportEfficiency * 0.05 + (abs(bindingAffinity) - 6) * 0.4))
    clearanceRate = 'LOW' if halfLifeHours >= 10 else 'MODERATE' if halfLifeHours >= 5 else 'HIGH'

    hydrogenBondCount = max(1, round(3 + solubility * 0.65))
    hydrophobicInteractionScore = max(0, min(100, 38 + abs(bindingAffinity) * 4.6))
    bindingPocketCoverage = max(35, min(98, 40 + targetReceptorBinding * 0.62))
    rmsdStability = max(0.7, min(3.4, 2.4 - (abs(bindingAffinity) - 6) * 0.16))

    hepatotoxicityValue = clamp(overallToxicity * 0.8 + (100 - solubility * 6) * 0.2)
    cardiotoxicityValue = clamp(overallToxicity * 0.7 + hydrophobicInteractionScore * 0.3)
    mutagenicityValue = clamp(overallToxicity * 0.65 + (100 - diffusionTrend) * 0.35)
    cytotoxicityValue = clamp(overallToxicity * 0.75 + (100 - cellUptakeRate) * 0.25)
    skinSensitivityValue = clamp(overallToxicity * 0.6 + (100 - responseTrend) * 0.4)

    drugLikenessScore = max(0, min(1, (100 - overallToxicity + efficacy) / 200))
    syntheticAccessibility = max(1, min(10, 2 + molecularWeight / 220 + abs(bindingAffinity) / 10))
    leadLikeness = 'PASS' if (molecularWeight < 450 and syntheticAccessibility < 6) else 'FAIL'

    predictionConfidence = max(45, min(99, multiFactor * 0.74 + 24))
    datasetSimilarityIndex = max(0.3, min(0.99, 0.42 + proteinInteraction / 180 + targetReceptorBinding / 220))
    modelUncertainty = 'LOW' if predictionConfidence >= 85 else 'MODERATE' if predictionConfidence >= 70 else 'HIGH'

    referenceDockingScore = -6.4
    compoundDockingScore = round(bindingAffinity, 2)
    dockingImprovementPercent = ((abs(compoundDockingScore) - abs(referenceDockingScore)) / abs(referenceDockingScore)) * 100

    return {
      'efficacy': round(efficacy, 1),
      'toxicity': round(overallToxicity, 1),
      'multiFactor': round(multiFactor, 1),
      'transportEfficiency': round(transportEfficiency, 1),
      'bioavailability': round(predictedBioavailability, 1),
      'dockingAffinity': round(bindingAffinity, 2),
      'pharmacodynamicsIndex': round((efficacy + targetReceptorBinding + proteinInteraction) / 3, 1),
      'decision': decision,
      'lipinskiCompliance': lipinskiCompliance,
      'cyp3a4InhibitionRisk': cypRisk,
      'waterSolubilityClass': waterSolubilityClass,
      'plasmaProteinBinding': round(plasmaProteinBinding, 1),
      'halfLifeHours': round(halfLifeHours, 1),
      'clearanceRate': clearanceRate,
      'hydrogenBondCount': hydrogenBondCount,
      'hydrophobicInteractionScore': round(hydrophobicInteractionScore, 1),
      'bindingPocketCoverage': round(bindingPocketCoverage, 1),
      'rmsdStability': round(rmsdStability, 2),
      'hepatotoxicityRisk': risk_band(hepatotoxicityValue),
      'cardiotoxicityRisk': risk_band(cardiotoxicityValue),
      'mutagenicityRisk': risk_band(mutagenicityValue),
      'cytotoxicityRisk': risk_band(cytotoxicityValue),
      'skinSensitivityRisk': risk_band(skinSensitivityValue),
      'drugLikenessScore': round(drugLikenessScore, 2),
      'syntheticAccessibility': round(syntheticAccessibility, 1),
      'leadLikeness': leadLikeness,
      'predictionConfidence': round(predictionConfidence, 1),
      'modelUncertainty': modelUncertainty,
      'datasetSimilarityIndex': round(datasetSimilarityIndex, 2),
      'referenceDockingScore': referenceDockingScore,
      'compoundDockingScore': compoundDockingScore,
      'dockingImprovementPercent': round(dockingImprovementPercent, 1),
    }


In [ ]:

#@title 5) State + UI controls
state = {
    'analysis_history': [],
    'prediction_history': [],
    'selected_analysis_id': None,
    'selected_analysis_mode': 'original',
    'history_search': '',
    'sample_name': 'candidate-001',
    'smiles': 'CC(=O)OC1=CC=CC=C1C(=O)O',
    'molecularWeight': 320.0,
    'bindingAffinity': -8.2,
    'solubility': 4.1,
    'cellUptakeRate': 65.0,
    'proteinInteraction': 70.0,
    'targetReceptorBinding': 72.0,
    'diffusionTrend': 74.0,
    'movementTrend': 71.0,
    'responseTrend': 69.0,
    'prediction': None,
}

analysis_upload = widgets.FileUpload(accept='.json', multiple=False, description='Upload Analysis JSON')
pred_upload = widgets.FileUpload(accept='.json', multiple=False, description='Upload Prediction JSON')
mock_count = widgets.IntSlider(value=6, min=1, max=30, step=1, description='Mock analyses')
btn_gen = widgets.Button(description='Generate Mock Analyses', button_style='info', icon='random')
btn_export_analysis = widgets.Button(description='Export Analysis JSON', button_style='success', icon='download')
btn_export_pred = widgets.Button(description='Export Prediction JSON', button_style='success', icon='download')

history_search = widgets.Text(description='History Search', placeholder='Search history sample name')
history_select = widgets.Dropdown(options=[], description='Analysis')
mode_select = widgets.Dropdown(options=[('Original Analysis','original'), ('Optimized Analysis','optimized')], value='original', description='Source')
btn_sync = widgets.Button(description='Run Sample Sync', icon='refresh')

sample_name = widgets.Text(description='Sample Name', value=state['sample_name'])
smiles = widgets.Text(description='SMILES', value=state['smiles'])
molecularWeight = widgets.FloatText(description='Mol Weight', value=state['molecularWeight'])
bindingAffinity = widgets.FloatText(description='Bind Affinity', value=state['bindingAffinity'])
solubility = widgets.FloatText(description='Solubility', value=state['solubility'])
cellUptakeRate = widgets.FloatText(description='Cell Uptake', value=state['cellUptakeRate'])
proteinInteraction = widgets.FloatText(description='Protein Int.', value=state['proteinInteraction'])
targetReceptorBinding = widgets.FloatText(description='Target Bind', value=state['targetReceptorBinding'])
diffusionTrend = widgets.FloatText(description='Diffusion T5', value=state['diffusionTrend'])
movementTrend = widgets.FloatText(description='Movement T5', value=state['movementTrend'])
responseTrend = widgets.FloatText(description='Response T5', value=state['responseTrend'])

btn_run = widgets.Button(description='Run Prediction', button_style='primary', icon='play')
btn_save_txt = widgets.Button(description='Save as Text', icon='download')
btn_save_pred = widgets.Button(description='Save Drug Prediction', button_style='warning', icon='save')

analysis_delete = widgets.Dropdown(options=[], description='Del Analysis')
btn_delete_analysis = widgets.Button(description='Delete Analysis', button_style='danger', icon='trash')
pred_delete = widgets.Dropdown(options=[], description='Del Prediction')
btn_delete_pred = widgets.Button(description='Delete Prediction', button_style='danger', icon='trash')

status = widgets.HTML('<b>Status:</b> Ready.')
out_diagram = widgets.Output()
out_prediction = widgets.Output()
out_history_tables = widgets.Output()


In [ ]:

#@title 6) App logic

def get_filtered_history():
    q = state['history_search'].strip().lower()
    if not q:
        return state['analysis_history']
    return [e for e in state['analysis_history'] if q in e.get('imageName','').lower()]


def refresh_dropdowns():
    filtered = get_filtered_history()
    opts = [(e['imageName'], e['id']) for e in filtered]

    if not opts:
        history_select.options = [('-- no match --', None)]
        history_select.value = None
        state['selected_analysis_id'] = None
    else:
        history_select.options = opts
        valid_ids = {v for _, v in opts}
        if state['selected_analysis_id'] not in valid_ids:
            state['selected_analysis_id'] = opts[0][1]
        history_select.value = state['selected_analysis_id']

    analysis_delete.options = [(e['imageName'], e['id']) for e in state['analysis_history']] or [('-- none --', None)]
    pred_delete.options = [(e['sampleName'], e['id']) for e in state['prediction_history']] or [('-- none --', None)]


def render_tables():
    with out_history_tables:
        clear_output(wait=True)

        display(Markdown('### Analysis History'))
        if not state['analysis_history']:
            display(Markdown('No analysis history available.'))
        else:
            rows = []
            for e in state['analysis_history']:
                rows.append({
                    'Image': e['imageName'],
                    'Date': e['createdAt'][:19],
                    'Risk': e['result']['screeningMetrics']['riskScore'],
                    'Final': e['result']['screeningMetrics']['finalScreeningScore'],
                    'Has Optimized': bool(e.get('optimizedResult')),
                })
            display(pd.DataFrame(rows))

        display(Markdown('### Prediction History'))
        if not state['prediction_history']:
            display(Markdown('No prediction history available.'))
        else:
            rows = []
            for e in state['prediction_history']:
                rows.append({
                    'Sample': e['sampleName'],
                    'Date': e['createdAt'][:19],
                    'Efficacy': e['outputs']['predictedEfficacy'],
                    'Toxicity': e['outputs']['predictiveToxicity'],
                    'Decision': e['outputs']['decision'],
                })
            display(pd.DataFrame(rows))


def rerender_all():
    refresh_dropdowns()
    render_tables()


def hydrate_from_analysis(analysis_id, mode):
    if not analysis_id:
        return False

    source = None
    for e in state['analysis_history']:
        if e['id'] == analysis_id:
            source = e
            break
    if not source:
        return False

    result = source.get('optimizedResult') if mode == 'optimized' and source.get('optimizedResult') else source.get('result')
    if not result or 'screeningMetrics' not in result:
        return False

    m = result['screeningMetrics']
    suffix = '-optimized' if mode == 'optimized' else ''

    state['sample_name'] = source['imageName'].rsplit('.', 1)[0] + suffix
    state['smiles'] = m.get('smiles') or 'CC(=O)OC1=CC=CC=C1C(=O)O'
    state['molecularWeight'] = to_safe_number(m.get('molecularWeight'), 320)
    state['bindingAffinity'] = to_safe_number(m.get('bindingAffinity'), -8.2)
    state['solubility'] = to_safe_number(m.get('solubility'), 4.1)
    state['cellUptakeRate'] = to_safe_number(m.get('cellUptakeRate'), 65)
    state['proteinInteraction'] = to_safe_number(m.get('proteinInteraction'), 70)
    state['targetReceptorBinding'] = to_safe_number(m.get('targetReceptorBinding'), 72)
    state['diffusionTrend'] = to_safe_number(m.get('diffusionTrend'), 74)
    state['movementTrend'] = to_safe_number(m.get('movementTrend'), 71)
    state['responseTrend'] = to_safe_number(m.get('responseTrend'), 69)

    sample_name.value = state['sample_name']
    smiles.value = state['smiles']
    molecularWeight.value = state['molecularWeight']
    bindingAffinity.value = state['bindingAffinity']
    solubility.value = state['solubility']
    cellUptakeRate.value = state['cellUptakeRate']
    proteinInteraction.value = state['proteinInteraction']
    targetReceptorBinding.value = state['targetReceptorBinding']
    diffusionTrend.value = state['diffusionTrend']
    movementTrend.value = state['movementTrend']
    responseTrend.value = state['responseTrend']

    return True


def render_smiles_diagram():
    with out_diagram:
        clear_output(wait=True)
        s = smiles.value.strip()
        if not s:
            display(Markdown('No SMILES entered.'))
            return

        pubchem = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{requests.utils.quote(s, safe='')}/PNG"
        cactus = f"https://cactus.nci.nih.gov/chemical/structure/{requests.utils.quote(s, safe='')}/image?format=png"

        ok = False
        for url in [pubchem, cactus]:
            try:
                r = requests.get(url, timeout=10)
                if r.ok and r.headers.get('content-type','').startswith('image'):
                    display(Markdown(f"2D Chemical Diagram source: `{url}`"))
                    img = PILImage.open(io.BytesIO(r.content))
                    display(img)
                    ok = True
                    break
            except Exception:
                pass

        if not ok:
            display(Markdown('Diagram unavailable from both providers.'))


def collect_inputs():
    return {
        'sampleName': sample_name.value,
        'smiles': smiles.value,
        'molecularWeight': molecularWeight.value,
        'bindingAffinity': bindingAffinity.value,
        'solubility': solubility.value,
        'cellUptakeRate': cellUptakeRate.value,
        'proteinInteraction': proteinInteraction.value,
        'targetReceptorBinding': targetReceptorBinding.value,
        'diffusionTrend': diffusionTrend.value,
        'movementTrend': movementTrend.value,
        'responseTrend': responseTrend.value,
    }


def render_prediction():
    with out_prediction:
        clear_output(wait=True)
        p = state['prediction']
        if not p:
            display(Markdown('Run prediction to see result sections.'))
            return

        display(Markdown(f"## Drug Prediction Result — {sample_name.value or 'Candidate'}"))

        display(Markdown(
            f"**Predicted efficacy:** {p['efficacy']}%  \
**Predictive toxicity score:** {p['toxicity']}%  \
**Multi-factor screening score:** {p['multiFactor']}  \
**Automated decision:** {p['decision']}  \
**Transport efficiency:** {p['transportEfficiency']}%  \
**Predicted bioavailability:** {p['bioavailability']}%"
        ))

        display(Markdown('### 1) ADMET Metrics'))
        display(Markdown(
            f"- Lipinski compliance: **{p['lipinskiCompliance']}**
"
            f"- CYP3A4 inhibition: **{p['cyp3a4InhibitionRisk']}**
"
            f"- Solubility class: **{p['waterSolubilityClass']}**
"
            f"- Plasma protein binding: **{p['plasmaProteinBinding']}%**
"
            f"- Half-life: **{p['halfLifeHours']} hours**
"
            f"- Clearance: **{p['clearanceRate']}**"
        ))

        display(Markdown('### 2) Molecular Interaction Metrics'))
        display(Markdown(
            f"- Hydrogen bonds: **{p['hydrogenBondCount']}**
"
            f"- Hydrophobic contacts score: **{p['hydrophobicInteractionScore']}**
"
            f"- Binding pocket coverage: **{p['bindingPocketCoverage']}%**
"
            f"- RMSD deviation: **{p['rmsdStability']} Å**"
        ))

        display(Markdown('### 3) Toxicity Sub-Scores'))
        display(Markdown(
            f"- Hepatotoxicity risk: **{p['hepatotoxicityRisk']}**
"
            f"- Cardiotoxicity risk: **{p['cardiotoxicityRisk']}**
"
            f"- Mutagenicity risk: **{p['mutagenicityRisk']}**
"
            f"- Cytotoxicity risk: **{p['cytotoxicityRisk']}**
"
            f"- Skin sensitivity risk: **{p['skinSensitivityRisk']}**"
        ))

        display(Markdown('### 4) Drug-likeness + 5) Confidence'))
        display(Markdown(
            f"- Drug-likeness score: **{p['drugLikenessScore']} / 1**
"
            f"- Synthetic accessibility: **{p['syntheticAccessibility']}**
"
            f"- Lead-likeness: **{p['leadLikeness']}**
"
            f"- Prediction confidence: **{p['predictionConfidence']}%**
"
            f"- Model uncertainty: **{p['modelUncertainty']}**
"
            f"- Dataset similarity index: **{p['datasetSimilarityIndex']}**"
        ))

        sign = '+' if p['dockingImprovementPercent'] > 0 else ''
        display(Markdown('### 6) Benchmark Comparison'))
        display(Markdown(
            f"- Reference drug docking score: **{p['referenceDockingScore']} kcal/mol**
"
            f"- Your compound docking score: **{p['compoundDockingScore']} kcal/mol**
"
            f"- Improvement vs reference: **{sign}{p['dockingImprovementPercent']}%**"
        ))

        display(Markdown('### Overall Scientific Evaluation'))
        display(Markdown('- Binding strength: ⭐⭐⭐⭐
- Bioavailability: ⭐⭐⭐⭐⭐
- Transport: ⭐⭐⭐⭐⭐
- Toxicity: ⭐⭐⭐
- Screening score: ⭐⭐⭐⭐'))


# ---- event handlers ----
def on_generate(_):
    n = int(mock_count.value)
    state['analysis_history'] = [make_history_entry(f'sample_{i+1:03d}.png', seed=100+i) for i in range(n)]
    state['selected_analysis_id'] = state['analysis_history'][0]['id'] if state['analysis_history'] else None
    rerender_all()
    hydrate_from_analysis(state['selected_analysis_id'], state['selected_analysis_mode'])
    render_smiles_diagram()
    status.value = f"<b>Status:</b> Generated {n} mock analysis records."


def on_analysis_upload(change):
    if not analysis_upload.value:
        return
    info = next(iter(analysis_upload.value.values()))
    try:
        parsed = json.loads(info['content'].decode('utf-8'))
        if not isinstance(parsed, list):
            raise ValueError('Analysis JSON must be a list.')
        state['analysis_history'] = parsed
        state['selected_analysis_id'] = parsed[0]['id'] if parsed else None
        rerender_all()
        status.value = f"<b>Status:</b> Loaded {len(parsed)} analysis records."
    except Exception as e:
        status.value = f"<b>Status:</b> Analysis upload failed: {e}"


def on_pred_upload(change):
    if not pred_upload.value:
        return
    info = next(iter(pred_upload.value.values()))
    try:
        parsed = json.loads(info['content'].decode('utf-8'))
        if not isinstance(parsed, list):
            raise ValueError('Prediction JSON must be a list.')
        state['prediction_history'] = parsed
        rerender_all()
        status.value = f"<b>Status:</b> Loaded {len(parsed)} prediction records."
    except Exception as e:
        status.value = f"<b>Status:</b> Prediction upload failed: {e}"


def on_export_analysis(_):
    with open('nanovision_analysis_history.json', 'w', encoding='utf-8') as f:
        json.dump(state['analysis_history'], f, indent=2)
    if IN_COLAB:
        files.download('nanovision_analysis_history.json')
    status.value = '<b>Status:</b> Exported nanovision_analysis_history.json.'


def on_export_pred(_):
    with open('nanovision_prediction_history.json', 'w', encoding='utf-8') as f:
        json.dump(state['prediction_history'], f, indent=2)
    if IN_COLAB:
        files.download('nanovision_prediction_history.json')
    status.value = '<b>Status:</b> Exported nanovision_prediction_history.json.'


def on_history_search(change):
    state['history_search'] = history_search.value
    refresh_dropdowns()


def on_select_analysis(change):
    state['selected_analysis_id'] = history_select.value


def on_mode_change(change):
    state['selected_analysis_mode'] = mode_select.value


def on_sync(_):
    synced = hydrate_from_analysis(state['selected_analysis_id'], state['selected_analysis_mode'])
    if not synced:
        status.value = '<b>Status:</b> Sample sync failed. Choose a valid analysis sample and source.'
        return
    render_smiles_diagram()
    status.value = f"<b>Status:</b> Sample synced from {state['selected_analysis_mode']} analysis."


def on_smiles_change(change):
    state['smiles'] = smiles.value
    render_smiles_diagram()


def on_run(_):
    inputs = collect_inputs()
    state['prediction'] = compute_prediction(inputs)
    render_prediction()
    status.value = f"<b>Status:</b> Prediction run complete for {inputs['sampleName'] or 'candidate'}."


def on_save_txt(_):
    p = state['prediction']
    if not p:
        status.value = '<b>Status:</b> Run prediction before exporting text.'
        return

    lines = [
      f"Sample: {sample_name.value}",
      f"SMILES: {smiles.value}",
      f"Predicted efficacy: {p['efficacy']}%",
      f"Predictive toxicity: {p['toxicity']}%",
      f"Decision: {p['decision']}",
      f"Lipinski: {p['lipinskiCompliance']}",
      f"CYP3A4 inhibition: {p['cyp3a4InhibitionRisk']}",
      f"Drug-likeness: {p['drugLikenessScore']}/1",
      f"Confidence: {p['predictionConfidence']}%",
      f"Reference docking: {p['referenceDockingScore']} kcal/mol",
      f"Compound docking: {p['compoundDockingScore']} kcal/mol",
      f"Improvement: {p['dockingImprovementPercent']}%",
    ]

    filename = f"{sample_name.value or 'drug-prediction'}.txt"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write('
'.join(lines))
    if IN_COLAB:
        files.download(filename)
    status.value = f"<b>Status:</b> Saved prediction text file: {filename}"


def on_save_prediction(_):
    p = state['prediction']
    if not p:
        status.value = '<b>Status:</b> Run prediction before saving history.'
        return

    record = {
        'id': str(uuid.uuid4()),
        'createdAt': datetime.utcnow().isoformat() + 'Z',
        'sampleName': sample_name.value or 'candidate',
        'smiles': smiles.value,
        'molecularWeight': molecularWeight.value,
        'bindingAffinity': bindingAffinity.value,
        'solubility': solubility.value,
        'cellUptakeRate': cellUptakeRate.value,
        'proteinInteraction': proteinInteraction.value,
        'targetReceptorBinding': targetReceptorBinding.value,
        'diffusionTrend': diffusionTrend.value,
        'movementTrend': movementTrend.value,
        'responseTrend': responseTrend.value,
        'outputs': {
            'predictedEfficacy': p['efficacy'],
            'predictiveToxicity': p['toxicity'],
            'multiFactorScore': p['multiFactor'],
            'decision': p['decision'],
            'lipinskiCompliance': p['lipinskiCompliance'],
            'cyp3a4InhibitionRisk': p['cyp3a4InhibitionRisk'],
            'waterSolubilityClass': p['waterSolubilityClass'],
            'plasmaProteinBinding': p['plasmaProteinBinding'],
            'halfLifeHours': p['halfLifeHours'],
            'clearanceRate': p['clearanceRate'],
            'hydrogenBondCount': p['hydrogenBondCount'],
            'hydrophobicInteractionScore': p['hydrophobicInteractionScore'],
            'bindingPocketCoverage': p['bindingPocketCoverage'],
            'rmsdStability': p['rmsdStability'],
            'hepatotoxicityRisk': p['hepatotoxicityRisk'],
            'cardiotoxicityRisk': p['cardiotoxicityRisk'],
            'mutagenicityRisk': p['mutagenicityRisk'],
            'cytotoxicityRisk': p['cytotoxicityRisk'],
            'skinSensitivityRisk': p['skinSensitivityRisk'],
            'drugLikenessScore': p['drugLikenessScore'],
            'syntheticAccessibility': p['syntheticAccessibility'],
            'leadLikeness': p['leadLikeness'],
            'predictionConfidence': p['predictionConfidence'],
            'modelUncertainty': p['modelUncertainty'],
            'datasetSimilarityIndex': p['datasetSimilarityIndex'],
            'referenceDockingScore': p['referenceDockingScore'],
            'compoundDockingScore': p['compoundDockingScore'],
            'dockingImprovementPercent': p['dockingImprovementPercent'],
        }
    }
    state['prediction_history'] = [record] + state['prediction_history'][:99]
    rerender_all()
    status.value = f"<b>Status:</b> Saved prediction for {record['sampleName']}."


def on_delete_analysis(_):
    target = analysis_delete.value
    if not target:
        status.value = '<b>Status:</b> No analysis item selected to delete.'
        return
    before = len(state['analysis_history'])
    state['analysis_history'] = [e for e in state['analysis_history'] if e['id'] != target]
    after = len(state['analysis_history'])
    if before == after:
        status.value = '<b>Status:</b> Analysis record not found.'
    else:
        if state['selected_analysis_id'] == target:
            state['selected_analysis_id'] = state['analysis_history'][0]['id'] if state['analysis_history'] else None
        rerender_all()
        status.value = '<b>Status:</b> Analysis record deleted.'


def on_delete_pred(_):
    target = pred_delete.value
    if not target:
        status.value = '<b>Status:</b> No prediction item selected to delete.'
        return
    before = len(state['prediction_history'])
    state['prediction_history'] = [e for e in state['prediction_history'] if e['id'] != target]
    after = len(state['prediction_history'])
    if before == after:
        status.value = '<b>Status:</b> Prediction record not found.'
    else:
        rerender_all()
        status.value = '<b>Status:</b> Prediction record deleted.'


# wire events
btn_gen.on_click(on_generate)
analysis_upload.observe(on_analysis_upload, names='value')
pred_upload.observe(on_pred_upload, names='value')
btn_export_analysis.on_click(on_export_analysis)
btn_export_pred.on_click(on_export_pred)
history_search.observe(on_history_search, names='value')
history_select.observe(on_select_analysis, names='value')
mode_select.observe(on_mode_change, names='value')
btn_sync.on_click(on_sync)
smiles.observe(on_smiles_change, names='value')
btn_run.on_click(on_run)
btn_save_txt.on_click(on_save_txt)
btn_save_pred.on_click(on_save_prediction)
btn_delete_analysis.on_click(on_delete_analysis)
btn_delete_pred.on_click(on_delete_pred)


In [ ]:

#@title 7) Launch Drug Discovery Analysis app
ui = widgets.VBox([
    widgets.HTML('<h2>Drug Discovery Analysis</h2><p>Multimodal workspace combining image-derived morphology, molecular descriptors, nano-bio interactions, and temporal dynamics for screening outputs.</p>'),

    widgets.HTML('<h4>Data IO</h4>'),
    widgets.HBox([analysis_upload, pred_upload, mock_count, btn_gen]),
    widgets.HBox([btn_export_analysis, btn_export_pred]),
    status,

    widgets.HTML('<hr><h4>Load Analysis from History</h4>'),
    history_search,
    widgets.HBox([history_select, mode_select, btn_sync]),

    widgets.HTML('<hr><h4>Inputs</h4>'),
    sample_name,
    smiles,
    out_diagram,
    widgets.HBox([molecularWeight, bindingAffinity, solubility]),
    widgets.HBox([cellUptakeRate, proteinInteraction, targetReceptorBinding]),
    widgets.HBox([diffusionTrend, movementTrend, responseTrend]),

    widgets.HBox([btn_run, btn_save_txt, btn_save_pred]),

    widgets.HTML('<hr><h4>Prediction Result</h4>'),
    out_prediction,

    widgets.HTML('<hr><h4>Quick Management</h4>'),
    widgets.HBox([analysis_delete, btn_delete_analysis]),
    widgets.HBox([pred_delete, btn_delete_pred]),
    out_history_tables,
])

display(ui)
on_generate(None)
render_smiles_diagram()
render_prediction()
